# p09_conflicting_evidence — when supervision ratings and the final evaluation disagree

A project is rated several times in its life, by different people, for different purposes. The agent and the analysts will meet all of these ratings in documents, and they often disagree. This notebook collects them for a sample of closed projects, **measures** how often they disagree, and implements a resolution policy that never hides the disagreement.

%md-sandbox
<pre class="mermaid">
---
config:
  layout: elk
---
flowchart LR
    ap["approval"]
    isr["mid-term to last ISR<br/>ISR ... ISR (isr_earlier) ... ISR (isr_last)<br/>task team: progress towards the PDO,<br/>implementation progress (forward-looking)"]
    cl["closing"]
    icr["ICR (self)<br/>outcome rating, Bank team<br/>(backward-looking, self-reported)"]
    ieg["IEG ICR Review<br/>outcome rating, independent"]
    ap -->|"time"| isr -->|"time"| cl -->|"time"| icr -->|"time"| ieg
</pre>
<script type="module">
  import mermaid from "https://cdn.jsdelivr.net/npm/mermaid@11.6.0/dist/mermaid.esm.min.mjs";
  import elkLayouts from "https://cdn.jsdelivr.net/npm/@mermaid-js/layout-elk@0.1.7/dist/mermaid-layout-elk.esm.min.mjs";
  mermaid.registerLayoutLoaders(elkLayouts);
  mermaid.initialize({ startOnLoad: true });
</script>

<details><summary>Text version of the diagram</summary>

```
 approval        mid-term                 last ISR        closing    ICR (self)       IEG ICR Review
    |    ISR ... ISR (isr_earlier) ... ISR (isr_last)        |       outcome rating    outcome rating
    |    task team: "progress towards the PDO",               |       Bank team         independent
    |    "implementation progress" (forward-looking)          |       (backward-looking, self-reported)
    +---------------------------- time ---------------------------------------------------------->
```

</details>

| Output | Grain | Key |
|---|---|---|
| `proc_bronze.rating_documents` | one document read per project and role (last ISR, earlier ISR, ICR Review), with the raw parsed ratings | `project_id, role` |
| `proc_silver.project_ratings_evidence` | one rating: project x source type x dimension, with score, date, independence and citation | `project_id, source_type, dimension` |
| `proc_gold.project_evidence_resolution` | the resolved answer per project and question type, with flag, confidence and rationale | `project_id, question_type` |

## Interface for the agent: `resolve_evidence()`

`from proc_intel.evidence import resolve_evidence` — a pure function, no Spark, unit-tested in `tests/test_evidence.py`.

**Input** `resolve_evidence(records, question_type, reference_date=None)`

| Field of each record | Type | Values |
|---|---|---|
| `project_id` | str | `P123456` (one project per call) |
| `source_type` | str | `ieg_validation`, `icr_self_rating`, `isr_last`, `isr_earlier` |
| `dimension` | str | `outcome`, `pdo_progress`, `implementation_progress` |
| `rating` | str | six-point label or abbreviation (`Moderately Satisfactory`, `MS`); the score is recomputed from it |
| `as_of` | date or ISO string | date of the rating |
| `doc_id`, `url`, `source` | str, optional | citation, returned unchanged |

`question_type` is `outcome` (how did the project do?) or `current_status` (how is implementation going?). `reference_date` (optional) is "today" for the staleness rule. Rows of `proc_silver.project_ratings_evidence` can be passed as they are: `[r.asDict() for r in rows]`.

**Output** (a dict)

| Key | Meaning |
|---|---|
| `answer` | the chosen record: `source_type, dimension, rating, score, as_of, independence, rank, specificity, doc_id, url, source` |
| `supporting` | other records within 1 point of the answer and on the same side of the satisfactory line (same fields, plus `points_from_answer`) |
| `conflicting` | records 2+ points away, or 1 point away across the satisfactory line |
| `superseded` | current_status only: ratings from older ISRs (history, not conflict) |
| `not_used` | records that cannot answer this question type, or carry no rating |
| `flag` | `consistent`, `disagreement` (1 point across the line), `conflict` (2+ points), `insufficient` |
| `confidence` | `high`, `medium`, `low` (rules in section 6) |
| `rationale` | one paragraph naming the chosen source and every conflicting one, with dates and document ids |

The resolved answers for the sampled projects are also stored in `proc_gold.project_evidence_resolution` (with the full result as JSON), so the agent can read them with SQL.

## How it was built and tested

**Step 1 — Where do the ratings live?**
- **Question:** which source gives the last ISR rating, the ICR self-rating and IEG's rating most cheaply and reliably?
- **What we found:** the outcome project's `bronze.ieg_ratings` has IEG's outcome for 6,906 ICR Reviews but **no ICR self-rating**, and its document chunks are only appraisal documents (no ISRs, no ICRs). The Projects API carries no ratings. The Documents API filters by document type (`docty_exact`), and every ISR and ICR Review has a plain-text rendering.
- **Decision:** list each project's ISRs, ICRs and ICR Reviews from the Documents API and read the ratings from the text. The ICR Review prints the ICR's rating and IEG's rating side by side, so one document gives both, and IEG's rating read from the text can be **checked against** `bronze.ieg_ratings` (a free accuracy test of the parser).

**Step 2 — The rating tables are fixed-width columns with wrapped cells**
- **Question:** can a line regex read the ratings?
- **What we found:** for ISRs mostly yes (`Progress towards achievement of PDO   <previous>   <current>`), but in ICR Reviews a cell such as "Moderately Satisfactory" is often split over two lines, *above and below* the row label, while the neighbouring cell sits on the label line. A line regex reads the wrong column.
- **Decision:** one parser for both tables: take the column positions from the header line, assign every word to the column under which it starts, and read each column's text top to bottom for rating phrases. Placeholders (`---`, `Not Rated`) are matched too, so an empty cell cannot shift the next row's rating up. The "Reason for disagreement" column is cut off, because its free text contains rating words.

**Step 3 — The first full run found two layout variants**
- **Question:** does the parser read the real documents, not just the probes?
- **What we found:** run 1 read IEG's outcome correctly for only 364 of 376 ICR Reviews, and for 91 of 400 projects the newest ISR had "no ratings". Two causes: (1) in some ICR Reviews the IEG cells start up to 5 characters **left** of the "IEG" title, so "Moderately" fell into the ICR column and IEG kept only "Satisfactory" (all 12 mismatches had that pattern); (2) ISRs from 2024 use a new template: the header says "Previous / Current" instead of "Previous Rating / Current Rating", and each rating is prefixed by an icon character from the Unicode private-use area. Falling back to an older ISR would have silently replaced the *last* ISR with an earlier one for a quarter of the sample.
- **Decision:** a column starts 8 characters before its title (but never before the second word of the previous column's rating), the header regex accepts both templates, and private-use characters are replaced by a space (which keeps the column offsets). Both layouts are now unit tests; section 4 re-checks IEG against the dataset.

**Step 4 — A bounded, unbiased sample**
- **Question:** which projects?
- **What we found:** 1,368 closed projects in the procurement data have an IEG outcome rating (evaluated FY2019-2027).
- **Decision:** 400 of them in hash order of `project_id` (reproducible and unrelated to the ratings, so the disagreement rates are not biased by the selection). For each: the last ISR, the ISR nearest the middle of the supervision period, and the ICR Review. If a document has no readable ratings block, the next candidate of the same role is tried and counted.

**Step 5 — Resolution as a policy, not a pick**
- **Question:** what should the agent say when sources disagree?
- **What we found:** the question decides which source is authoritative. For "how did the project do", IEG's independent ex-post rating is the Bank's official verdict; for "how is it going now", only the latest ISR speaks to the present.
- **Decision:** a documented hierarchy per question type, every other record returned as supporting or conflicting with its date and citation, a flag and a confidence label. Tested on hand-made cases in `tests/test_evidence.py`.

In [ ]:
%pip install -q rapidfuzz unidecode

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [ ]:
dbutils.library.restartPython()

In [ ]:
%run ./proc_common

## Settings

**What:** the sample size, the Documents API address and document types, the folders for raw files, and the table names.

**Why:** raw listings and texts are kept in the bronze volume, so the parse can be re-run or audited without calling the API again, and a restart skips everything already saved.

In [ ]:
import json
import random
import time
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timezone

import requests
from pyspark.sql import Window

from proc_intel.evidence import HIERARCHY, SCALE, SHORT, parse_icrr_ratings, parse_isr_ratings, resolve_evidence
from proc_intel.http import RETRYABLE_STATUS, get_json, retry_delay
from proc_intel.similarity import wilson_interval

SAMPLE_SIZE = 400
DOCS_URL = "https://search.worldbank.org/api/v3/wds"
DOC_TYPES = {"isr": "Implementation Status and Results Report",
             "icr": "Implementation Completion and Results Report",
             "icrr": "Implementation Completion Report Review"}
MAX_TRIES_PER_ROLE = 3
EVIDENCE_DIR = f"{RAW_VOLUME}/evidence"
DOCS_TABLE = f"{BRONZE}.rating_documents"
EVIDENCE_TABLE = f"{SILVER}.project_ratings_evidence"
RESOLUTION_TABLE = f"{GOLD}.project_evidence_resolution"
IEG_RATINGS = f"{CATALOG}.bronze.ieg_ratings"           # outcome project, read only
for sub in ["lists", "docs"]:
    os.makedirs(f"{EVIDENCE_DIR}/{sub}", exist_ok=True)

## 1. The project sample

**What:** closed projects of the procurement model (`proc_silver.dim_project`) with an IEG ICR Review outcome other than "Not Rated", ordered by a hash of the project id; the first 400.

**Why:** these projects have both procurement data (contracts, notices) and an independent verdict, so the evidence can later be joined to the procurement graph. Hash order is reproducible and independent of the ratings.

In [ ]:
ieg = (spark.table(IEG_RATINGS).filter("evaluation_type = 'ICRR' AND outcome <> 'Not Rated'")
       .withColumn("_rn", F.row_number().over(Window.partitionBy("project_id").orderBy(F.desc("evaluation_fy"))))
       .filter("_rn = 1").select("project_id", F.col("outcome").alias("ieg_outcome_dataset"),
                                 F.col("evaluation_fy").cast("int").alias("evaluation_fy")))
eligible = (spark.table(f"{SILVER}.dim_project").filter("status = 'Closed'")
            .join(ieg, "project_id").select("project_id", "project_name", "region", "major_sector", "country_code",
                                            "approval_fy", "ieg_outcome_dataset", "evaluation_fy"))
sample = eligible.orderBy(F.sha2("project_id", 256)).limit(SAMPLE_SIZE).cache()
print(f"eligible closed projects with an IEG outcome: {eligible.count():,}; sampled: {sample.count()}")
display(sample.groupBy("region").count().orderBy(F.desc("count")))

eligible closed projects with an IEG outcome: 1,368; sampled: 400


region                                               | count
-----------------------------------------------------+------
Western and Central Africa                           | 87   
Eastern and Southern Africa                          | 68   
East Asia and Pacific                                | 64   
Latin America and Caribbean                          | 48   
South Asia                                           | 48   
Europe and Central Asia                              | 45   
Middle East, North Africa, Afghanistan, and Pakistan | 40   

## 2. Find and read the documents (resumable, 4 projects in parallel)

**What:** for each project, `collect_project()`
1. lists its ISRs, ICRs and ICR Reviews (one Documents API call per type; the listing is saved as JSON),
2. reads the **last ISR** (newest first; if a document has no readable ratings block, the next older one, up to 3),
3. reads the **earlier ISR** nearest the middle of the supervision period (first ISR to last ISR),
4. reads the newest **ICR Review** with a ratings table,

saving every text file to the volume, and returns one small row per role with the parsed ratings. Only the ICR's date is needed (its rating is printed in the ICR Review), so ICRs are listed but not downloaded.

**Why:** three small documents per project instead of all of them. Saved listings and texts make a re-run free and let anyone re-check a parsed rating against its source text.

In [ ]:
HTTP = requests.Session()


def get_text(url: str, max_retries: int = 5) -> str | None:
    """GET a document's text rendering with the same retry rules as get_json; None if the server has no text
    for it (404/403/410: some listed documents have no text rendering)."""
    for attempt in range(max_retries + 1):
        try:
            resp = HTTP.get(url, timeout=120)
            if resp.status_code in (403, 404, 410):
                return None
            if resp.status_code not in RETRYABLE_STATUS:
                resp.raise_for_status()
                return resp.content.decode("utf-8", errors="replace")
            delay = retry_delay(attempt, resp.text, resp.headers.get("Retry-After"))
        except (requests.ConnectionError, requests.Timeout):
            delay = retry_delay(attempt)
        time.sleep(delay + random.uniform(0, 1))
    raise RuntimeError(f"giving up on {url}")


def list_documents(project_id: str) -> dict:
    path = f"{EVIDENCE_DIR}/lists/{project_id}.json"
    if os.path.exists(path):
        with open(path) as f:
            return json.load(f)
    listing = {}
    for role, docty in DOC_TYPES.items():
        body = get_json(DOCS_URL, {"format": "json", "proid": project_id, "docty_exact": docty, "rows": 100,
                                   "fl": "id,docdt,docty,txturl,url"}, session=HTTP)
        docs = [d for k, d in body.get("documents", {}).items() if k != "facets" and d.get("txturl")]
        listing[role] = sorted(docs, key=lambda d: d.get("docdt") or "", reverse=True)     # newest first
    with open(path, "w") as f:
        json.dump(listing, f)
    return listing


def read_document(doc: dict) -> tuple[str | None, str | None]:
    """The saved text of a document (fetched once). A '.missing' marker remembers documents without text."""
    path, missing = f"{EVIDENCE_DIR}/docs/{doc['id']}.txt", f"{EVIDENCE_DIR}/docs/{doc['id']}.missing"
    if os.path.exists(missing):
        return None, None
    if not os.path.exists(path):
        text = get_text(doc["txturl"])
        if text is None:
            open(missing, "w").close()
            return None, None
        with open(path, "w", encoding="utf-8") as f:
            f.write(text)
    with open(path, encoding="utf-8") as f:
        return f.read(), path


def first_readable(candidates: list[dict], parse, has_rating) -> tuple[dict | None, dict, str | None, int, int]:
    """First candidate (in the given order) whose parsed ratings are usable; also how many were tried."""
    for tried, doc in enumerate(candidates[:MAX_TRIES_PER_ROLE], start=1):
        text, path = read_document(doc)
        parsed = parse(text)
        if text and has_rating(parsed):
            return doc, parsed, path, tried, len(text)
    return None, {}, None, min(len(candidates), MAX_TRIES_PER_ROLE), 0


def _date(doc: dict | None):
    return datetime.fromisoformat(doc["docdt"][:10]).date() if doc and doc.get("docdt") else None


def collect_project(project_id: str) -> list[dict]:
    listing = list_documents(project_id)
    isrs, icrs, icrrs = listing["isr"], listing["icr"], listing["icrr"]
    isr_ok = lambda p: bool(p.get("pdo_current") or p.get("ip_current"))
    rows = []

    def add(role, doc, parsed, path, tried, n_chars, n_candidates):
        rows.append({"project_id": project_id, "role": role, "doc_id": doc["id"] if doc else None,
                     "doc_date": _date(doc), "txt_url": doc["txturl"] if doc else None, "file_path": path,
                     "n_chars": n_chars, "n_candidates": n_candidates, "n_tried": tried,
                     **{k: (v.isoformat() if hasattr(v, "isoformat") else v) for k, v in parsed.items()}})

    last, parsed, path, tried, n = first_readable(isrs, parse_isr_ratings, isr_ok)
    add("isr_last", last, parsed, path, tried, n, len(isrs))
    if last:
        older = [d for d in isrs if (d.get("docdt") or "") < last["docdt"]]
        if older:
            first_d, last_d = _date(older[-1]), _date(last)
            middle = first_d + (last_d - first_d) / 2
            by_distance = sorted(older, key=lambda d: abs((_date(d) - middle).days))
            mid, parsed, path, tried, n = first_readable(by_distance, parse_isr_ratings, isr_ok)
            add("isr_earlier", mid, parsed, path, tried, n, len(older))
    icrr, parsed, path, tried, n = first_readable(icrrs, parse_icrr_ratings,
                                                  lambda p: bool(p.get("ieg_outcome") or p.get("icr_outcome")))
    add("icrr", icrr, {**parsed, "icr_doc_id": icrs[0]["id"] if icrs else None,
                       "icr_doc_date": _date(icrs[0]).isoformat() if icrs else None}, path, tried, n, len(icrrs))
    return rows


started = time.time()
project_ids = [r.project_id for r in sample.select("project_id").collect()]
with ThreadPoolExecutor(max_workers=4) as pool:
    collected = [row for rows in pool.map(collect_project, project_ids) for row in rows]
print(f"{len(project_ids)} projects, {len(collected)} role rows in {time.time() - started:.0f} s")

400 projects, 1200 role rows in 24 s


## 3. Bronze: `proc_bronze.rating_documents`

**What:** one row per project and role with the document read, its date, file path, how many candidates existed and were tried, and the raw parsed ratings.

**Why:** the audit trail between the source text and the silver evidence. Every rating in silver can be traced to a file in the volume and a URL.

In [ ]:
doc_schema = ("project_id string, role string, doc_id string, doc_date date, txt_url string, file_path string, "
              "n_chars int, n_candidates int, n_tried int, seq_no int, archived_on string, pdo_current string, "
              "pdo_previous string, ip_current string, ip_previous string, icr_outcome string, ieg_outcome string, "
              "icr_bank_performance string, ieg_bank_performance string, icr_doc_id string, icr_doc_date string")
fields = [f.split()[0] for f in doc_schema.split(", ")]
docs_df = (spark.createDataFrame([{k: r.get(k) for k in fields} for r in collected], doc_schema)
           .withColumn("archived_on", F.to_date("archived_on")).withColumn("icr_doc_date", F.to_date("icr_doc_date"))
           .withColumn("_fetched_at", F.current_timestamp()))
n_docs = publish_table(
    docs_df, DOCS_TABLE,
    "Rating documents read for the evidence sample (p09). Grain: one row per project and role (isr_last, isr_earlier, "
    "icrr). Source: World Bank Documents API (search.worldbank.org/api/v3/wds), text renderings saved in "
    "proc_bronze.raw/evidence.",
    {"project_id": "PK. Project id", "role": "PK. isr_last, isr_earlier or icrr",
     "doc_id": "Documents API id of the document read (null: none readable)", "doc_date": "Document date (docdt)",
     "txt_url": "Text rendering URL", "file_path": "Saved text in the bronze volume",
     "n_chars": "Characters in the text", "n_candidates": "Documents of this role available for the project",
     "n_tried": "Documents read until one had a readable ratings table",
     "seq_no": "ISR sequence number", "archived_on": "ISR archive date (the rating date)",
     "pdo_current": "ISR: progress towards the PDO, this ISR", "pdo_previous": "ISR: same, previous ISR",
     "ip_current": "ISR: overall implementation progress, this ISR", "ip_previous": "ISR: same, previous ISR",
     "icr_outcome": "ICR Review table: outcome as rated by the ICR (self)", "ieg_outcome": "ICR Review table: outcome as rated by IEG",
     "icr_bank_performance": "ICR Review table: Bank performance, ICR", "ieg_bank_performance": "ICR Review table: Bank performance, IEG",
     "icr_doc_id": "Newest ICR document id (date source for the self-rating)", "icr_doc_date": "Its date",
     "_fetched_at": "Load time"},
    primary_key=["project_id", "role"])
D = spark.table(DOCS_TABLE)
display(D.groupBy("role").agg(F.count("*").alias("projects"), F.count("doc_id").alias("with_readable_doc"),
                              F.sum(F.when(F.col("n_candidates") == 0, 1).otherwise(0)).alias("no_document_listed"),
                              F.sum(F.when(F.col("n_tried") > 1, 1).otherwise(0)).alias("needed_fallback"),
                              F.round(F.avg("n_chars")).alias("avg_chars")).orderBy("role"))

role        | projects | with_readable_doc | no_document_listed | needed_fallback | avg_chars
------------+----------+-------------------+--------------------+-----------------+----------
icrr        | 400      | 381               | 2                  | 1               | 57259.0  
isr_earlier | 400      | 400               | 0                  | 1               | 23523.0  
isr_last    | 400      | 400               | 0                  | 3               | 24877.0  

## 4. Parser check against IEG's own dataset

**What:** compares IEG's outcome as read from the ICR Review text with IEG's outcome in `bronze.ieg_ratings`, and lists the disagreements.

**Why:** the dataset is an independent ground truth for one of the four columns the parser reads. A high match rate on IEG's column is evidence that the column assignment (and therefore the ICR column next to it) works; the mismatches show what the parser gets wrong.

In [ ]:
check = (D.filter("role = 'icrr' AND ieg_outcome IS NOT NULL").join(sample.select("project_id", "ieg_outcome_dataset"), "project_id"))
n_check, n_match = check.count(), check.filter("ieg_outcome = ieg_outcome_dataset").count()
low, high = wilson_interval(n_match, n_check)
print(f"IEG outcome parsed from the ICR Review = IEG dataset: {n_match} of {n_check} ({n_match / n_check:.1%}, "
      f"95% CI {low:.1%}-{high:.1%})")
display(check.filter("ieg_outcome <> ieg_outcome_dataset")
        .select("project_id", "doc_id", "ieg_outcome", "ieg_outcome_dataset", "icr_outcome").limit(15))

IEG outcome parsed from the ICR Review = IEG dataset: 381 of 381 (100.0%, 95% CI 99.0%-100.0%)


project_id | doc_id | ieg_outcome | ieg_outcome_dataset | icr_outcome
-----------+--------+-------------+---------------------+------------

## 5. Silver: `proc_silver.project_ratings_evidence`

**What:** one row per rating, in the long format `resolve_evidence()` takes:

| source_type | dimension | rating from | as_of | citation |
|---|---|---|---|---|
| `ieg_validation` | outcome | `bronze.ieg_ratings` (IEG's dataset; the text parse only cross-checks it) | ICR Review date (else 30 June of the evaluation FY) | ICR Review document |
| `icr_self_rating` | outcome | ICR column of the ICR Review table | ICR date (else ICR Review date) | ICR Review document (where it was read) |
| `isr_last`, `isr_earlier` | pdo_progress, implementation_progress | "Overall Ratings" block, current column | ISR archive date (else document date) | ISR document |

**Why:** a long table holds any number of sources and dimensions without schema changes; adding Aide-Memoire or restructuring-paper ratings later is new rows, not new columns. Every row carries its independence and a citation, which is what the resolution policy and the agent need.

In [ ]:
score_map = F.create_map(*[x for k, v in SCALE.items() for x in (F.lit(k), F.lit(v))])
doc_url = lambda c: F.regexp_replace(F.col(c), r"/text/.*$", "")         # the document page, not the text file
icrr = D.filter("role = 'icrr'").alias("r")
isr = D.filter("role IN ('isr_last', 'isr_earlier') AND doc_id IS NOT NULL")
ieg_rows = (sample.select("project_id", "ieg_outcome_dataset", "evaluation_fy").join(icrr, "project_id", "left")
            .select("project_id", F.lit("ieg_validation").alias("source_type"), F.lit("outcome").alias("dimension"),
                    F.col("ieg_outcome_dataset").alias("rating"),
                    F.coalesce("doc_date", F.make_date("evaluation_fy", F.lit(6), F.lit(30))).alias("as_of"),
                    F.concat(F.lit("IEG ICR Review, evaluation FY"), F.col("evaluation_fy").cast("string")).alias("source"),
                    F.coalesce("doc_id", F.lit("bronze.ieg_ratings")).alias("doc_id"), doc_url("txt_url").alias("doc_url")))
icr_rows = (icrr.filter("icr_outcome IS NOT NULL")
            .select("project_id", F.lit("icr_self_rating").alias("source_type"), F.lit("outcome").alias("dimension"),
                    F.col("icr_outcome").alias("rating"), F.coalesce("icr_doc_date", "doc_date").alias("as_of"),
                    F.concat(F.lit("ICR self-rating as printed in the ICR Review (ICR document "),
                             F.coalesce("icr_doc_id", F.lit("not listed")), F.lit(")")).alias("source"),
                    "doc_id", doc_url("txt_url").alias("doc_url")))
isr_rows = (isr.select("project_id", F.col("role").alias("source_type"),
                       F.explode(F.array(F.struct(F.lit("pdo_progress").alias("dimension"), F.col("pdo_current").alias("rating")),
                                         F.struct(F.lit("implementation_progress").alias("dimension"), F.col("ip_current").alias("rating")))).alias("r"),
                       F.coalesce("archived_on", "doc_date").alias("as_of"),
                       F.concat(F.lit("ISR sequence "), F.coalesce(F.col("seq_no").cast("string"), F.lit("?"))).alias("source"),
                       "doc_id", doc_url("txt_url").alias("doc_url"))
            .select("project_id", "source_type", "r.dimension", "r.rating", "as_of", "source", "doc_id", "doc_url")
            .filter("rating IS NOT NULL"))
evidence = (ieg_rows.unionByName(icr_rows).unionByName(isr_rows)
            .withColumn("score", F.try_element_at(score_map, F.col("rating")).cast("int"))
            .withColumn("is_satisfactory", F.col("score") >= 4)
            .withColumn("independence", F.when(F.col("source_type") == "ieg_validation", "independent").otherwise("self_reported")))
n_evidence = publish_table(
    evidence, EVIDENCE_TABLE,
    "Project performance ratings from several sources (p09). Grain: one row per project x source_type x dimension. "
    "Scores: six-point scale, Highly Unsatisfactory 1 ... Highly Satisfactory 6. Input format of proc_intel.evidence.resolve_evidence().",
    {"project_id": "PK, FK dim_project", "source_type": "PK. ieg_validation, icr_self_rating, isr_last, isr_earlier",
     "dimension": "PK. outcome (ICR, IEG), pdo_progress or implementation_progress (ISR)",
     "rating": "Six-point rating label", "as_of": "Date of the rating",
     "source": "Human-readable source (ISR sequence number, evaluation FY, ICR document id)",
     "doc_id": "Documents API id of the document the rating was read from (IEG: the ICR Review)",
     "doc_url": "Document page on documents.worldbank.org", "score": "1 (Highly Unsatisfactory) to 6 (Highly Satisfactory)",
     "is_satisfactory": "score >= 4 (Moderately Satisfactory or better)",
     "independence": "independent (IEG) or self_reported (task team, ICR)"},
    primary_key=["project_id", "source_type", "dimension"])
add_foreign_key(EVIDENCE_TABLE, ["project_id"], f"{SILVER}.dim_project", ["project_id"])
E = spark.table(EVIDENCE_TABLE)
display(dq_report([
    (EVIDENCE_TABLE, "rows", n_evidence, "info"),
    (EVIDENCE_TABLE, "duplicate keys", duplicate_keys(E, ["project_id", "source_type", "dimension"]), "== 0"),
    (EVIDENCE_TABLE, "ratings outside the six-point scale", E.filter("score IS NULL").count(), "== 0"),
    (EVIDENCE_TABLE, "rows without a date", null_count(E, "as_of"), "== 0"),
    (EVIDENCE_TABLE, "orphan project_id", orphan_count(E, "project_id", spark.table(f"{SILVER}.dim_project"), "project_id"), "== 0"),
]))
display(E.groupBy("source_type", "dimension").agg(F.countDistinct("project_id").alias("projects"),
                                                 F.round(F.avg("score"), 2).alias("mean_score"),
                                                 F.round(F.avg(F.col("is_satisfactory").cast("int")), 3).alias("share_satisfactory"))
        .orderBy("source_type", "dimension"))

table                                               | check                               | value | rule | status
----------------------------------------------------+-------------------------------------+-------+------+-------
dbw_wb_outcome.proc_silver.project_ratings_evidence | rows                                | 2381  | info | info  
dbw_wb_outcome.proc_silver.project_ratings_evidence | duplicate keys                      | 0     | == 0 | pass  
dbw_wb_outcome.proc_silver.project_ratings_evidence | ratings outside the six-point scale | 0     | == 0 | pass  
dbw_wb_outcome.proc_silver.project_ratings_evidence | rows without a date                 | 0     | == 0 | pass  
dbw_wb_outcome.proc_silver.project_ratings_evidence | orphan project_id                   | 0     | == 0 | pass  

source_type     | dimension               | projects | mean_score | share_satisfactory
----------------+-------------------------+----------+------------+-------------------
icr_self_rating | outcome                 | 381      | 4.49       | 0.869             
ieg_validation  | outcome                 | 400      | 4.27       | 0.83              
isr_earlier     | implementation_progress | 400      | 3.99       | 0.785             
isr_earlier     | pdo_progress            | 400      | 4.21       | 0.858             
isr_last        | implementation_progress | 400      | 4.38       | 0.903             
isr_last        | pdo_progress            | 400      | 4.5        | 0.908             

## 6. How often do the sources disagree?

**What:** one wide row per project (last ISR PDO and IP, earlier ISR PDO, ICR, IEG), then
- the **confusion matrix** of the last ISR's PDO rating against IEG's outcome,
- the **disconnect**: the last ISR rated progress Moderately Satisfactory or better, IEG rated the outcome Moderately Unsatisfactory or worse, as a share of (a) satisfactory last ISRs and (b) projects IEG rated unsatisfactory,
- the same for the ICR self-rating and the mid-period ISR, and by region and sector.

**Why:** the Bank's own evaluators have long reported that supervision ratings are more optimistic than final evaluations. Here it is measured on this sample, with confidence intervals, so the resolution policy rests on numbers rather than folklore.

In [ ]:
wide = (E.withColumn("k", F.concat_ws("|", "source_type", "dimension")).groupBy("project_id")
        .pivot("k", ["isr_last|pdo_progress", "isr_last|implementation_progress", "isr_earlier|pdo_progress",
                     "icr_self_rating|outcome", "ieg_validation|outcome"]).agg(F.first("score"))
        .toDF("project_id", "isr_pdo", "isr_ip", "mid_pdo", "icr", "ieg")
        .join(sample.select("project_id", "region", "major_sector", "evaluation_fy"), "project_id").cache())
pdf = wide.toPandas()            # 400 rows
both = pdf.dropna(subset=["isr_pdo", "ieg"])
SCORE_SHORT = {v: SHORT[k] for k, v in SCALE.items()}          # 6 -> 'HS', ..., 1 -> 'HU'
labels = [SCORE_SHORT[v] for v in range(6, 0, -1)]
matrix = (pd.crosstab(both["isr_pdo"].astype(int).map(SCORE_SHORT), both["ieg"].astype(int).map(SCORE_SHORT))
          .reindex(index=labels, columns=labels, fill_value=0))
matrix.index.name, matrix.columns.name = "last ISR PDO rating", "IEG outcome"
print(f"projects with both a last-ISR PDO rating and an IEG outcome: {len(both)}")
display(matrix.reset_index())

projects with both a last-ISR PDO rating and an IEG outcome: 400


last ISR PDO rating | HS | S   | MS | MU | U | HU
--------------------+----+-----+----+----+---+---
HS                  | 6  | 10  | 3  | 0  | 0 | 0 
S                   | 18 | 122 | 68 | 5  | 0 | 0 
MS                  | 2  | 17  | 82 | 27 | 2 | 1 
MU                  | 0  | 0   | 4  | 16 | 6 | 0 
U                   | 0  | 0   | 0  | 0  | 5 | 4 
HU                  | 0  | 0   | 0  | 0  | 0 | 2 

In [ ]:
def rate_row(df, earlier: str, label: str) -> dict:
    """Disconnect between an earlier self-rating column and IEG, with Wilson intervals."""
    d = df.dropna(subset=[earlier, "ieg"])
    sat, ieg_unsat = d[d[earlier] >= 4], d[d["ieg"] < 4]
    n_dis = int(((d[earlier] >= 4) & (d["ieg"] < 4)).sum())
    lo1, hi1 = wilson_interval(n_dis, len(sat))
    lo2, hi2 = wilson_interval(n_dis, len(ieg_unsat))
    gap = d[earlier] - d["ieg"]
    return {"earlier rating": label, "projects": len(d), "rated satisfactory": len(sat),
            "IEG unsatisfactory": len(ieg_unsat), "disconnects": n_dis,
            "share of satisfactory ratings that IEG rated unsatisfactory": f"{n_dis / max(len(sat), 1):.1%} ({lo1:.1%}-{hi1:.1%})",
            "share of IEG-unsatisfactory projects still rated satisfactory": f"{n_dis / max(len(ieg_unsat), 1):.1%} ({lo2:.1%}-{hi2:.1%})",
            "mean score gap (earlier - IEG)": round(float(gap.mean()), 2),
            "2+ points apart": int((gap.abs() >= 2).sum()), "earlier lower than IEG": int((gap < 0).sum())}


display(pd.DataFrame([rate_row(pdf, "mid_pdo", "mid-period ISR, PDO"), rate_row(pdf, "isr_pdo", "last ISR, PDO"),
                      rate_row(pdf, "isr_ip", "last ISR, implementation"), rate_row(pdf, "icr", "ICR self-rating")]))

earlier rating           | projects | rated satisfactory | IEG unsatisfactory | disconnects | share of satisfactory ratings that IEG rated unsatisfactory | share of IEG-unsatisfactory projects still rated satisfactory | mean score gap (earlier - IEG) | 2+ points apart | earlier lower than IEG
-------------------------+----------+--------------------+--------------------+-------------+-------------------------------------------------------------+---------------------------------------------------------------+--------------------------------+-----------------+-----------------------
mid-period ISR, PDO      | 400      | 343                | 68                 | 50          | 14.6% (11.2%-18.7%)                                         | 73.5% (62.0%-82.6%)                                           | -0.06                          | 51              | 132                   
last ISR, PDO            | 400      | 363                | 68                 | 35          | 9.6% (7.0%-13.1%)       

**What:** the last-ISR disconnect by region and by major sector (groups with at least 15 projects).

**Why:** if the disconnect concentrates in some regions or sectors, the agent should be more sceptical of supervision ratings there. With 400 projects the groups are small, so the intervals are wide and are printed with the rates.

In [ ]:
def by_group(col: str) -> pd.DataFrame:
    d = pdf.dropna(subset=["isr_pdo", "ieg"])
    rows = []
    for key, g in d.groupby(col):
        if len(g) < 15:
            continue
        n_dis = int(((g["isr_pdo"] >= 4) & (g["ieg"] < 4)).sum())
        n_unsat = int((g["ieg"] < 4).sum())
        lo, hi = wilson_interval(n_dis, int((g["isr_pdo"] >= 4).sum()))
        rows.append({col: key, "projects": len(g), "IEG unsatisfactory": n_unsat,
                     "last ISR satisfactory but IEG unsatisfactory": n_dis,
                     "disconnect rate (of satisfactory ISRs)": f"{n_dis / max(int((g['isr_pdo'] >= 4).sum()), 1):.1%} ({lo:.0%}-{hi:.0%})",
                     "IEG-unsatisfactory missed by the last ISR": f"{n_dis}/{n_unsat}"})
    return pd.DataFrame(rows).sort_values("projects", ascending=False)


display(by_group("region"))
display(by_group("major_sector"))

region                                               | projects | IEG unsatisfactory | last ISR satisfactory but IEG unsatisfactory | disconnect rate (of satisfactory ISRs) | IEG-unsatisfactory missed by the last ISR
-----------------------------------------------------+----------+--------------------+----------------------------------------------+----------------------------------------+------------------------------------------
Western and Central Africa                           | 87       | 12                 | 6                                            | 7.4% (3%-15%)                          | 6/12                                     
Eastern and Southern Africa                          | 68       | 11                 | 7                                            | 10.9% (5%-21%)                         | 7/11                                     
East Asia and Pacific                                | 64       | 12                 | 5                                            

major_sector                                       | projects | IEG unsatisfactory | last ISR satisfactory but IEG unsatisfactory | disconnect rate (of satisfactory ISRs) | IEG-unsatisfactory missed by the last ISR
---------------------------------------------------+----------+--------------------+----------------------------------------------+----------------------------------------+------------------------------------------
FY17 - Agriculture                                 | 94       | 19                 | 12                                           | 13.8% (8%-23%)                         | 12/19                                    
FY17 - Education                                   | 56       | 7                  | 3                                            | 5.9% (2%-16%)                          | 3/7                                      
FY17 - Energy and Extractives                      | 51       | 12                 | 6                                            | 13.3% (6

**What the numbers say (run of 2026-10-20, 400 projects).**
- The last ISR is more optimistic than IEG: 90.8% of last ISRs rate PDO progress satisfactory, IEG rates 83.0% of outcomes satisfactory.
- Of the 68 projects IEG rated unsatisfactory, **35 (51%) still had a satisfactory PDO rating in their last ISR**; 9.6% of satisfactory last ISRs ended unsatisfactory. Only 4 projects went the other way (last ISR unsatisfactory, IEG satisfactory).
- Mid-period ISRs are noisier in both directions (51 projects 2+ points from IEG, 132 rated lower than IEG): they reflect implementation trouble that is later fixed as often as trouble that persists.
- The ICR is closer to IEG but still self-favourable: IEG rated the outcome lower than the ICR for 91 of 381 projects and higher for 6; 17 of 66 unsatisfactory outcomes (26%) were rated satisfactory by the ICR.
- Regional and sector rates (7%-15%) overlap within their intervals at this sample size; no group can be singled out with confidence. Agriculture has the most disconnects (12 of its 19 unsatisfactory outcomes).

The consequence for the agent: a satisfactory supervision rating is weak evidence of a good outcome; for an outcome question it must defer to IEG, and where IEG has not rated yet it must say so.

## 7. The resolution policy

**Evidence attributes.** Every record carries four attributes the policy uses:

| Attribute | Values | Used for |
|---|---|---|
| source type | ieg_validation, icr_self_rating, isr_last, isr_earlier | rank in the hierarchy |
| independence | independent (IEG) / self-reported (task team, ICR) | confidence; shown to the user |
| recency | `as_of` date | tie-break within a rank; staleness of a status |
| specificity | direct (rates what is asked) / proxy (a related rating, e.g. ISR progress for an outcome question) | confidence |

**Hierarchy per question type** (best first; within a rank the newest record wins):

| Question | 1 | 2 | 3 | 4 |
|---|---|---|---|---|
| outcome ("did it achieve its objectives?") | IEG validation | ICR self-rating | last ISR, PDO (proxy) | earlier ISR, PDO (proxy) |
| current status ("how is implementation going?") | newest ISR, implementation progress | newest ISR, PDO progress | older ISRs (history only) | - |

**Rules.**
1. **Never pick silently.** The answer comes with every other usable record, each labelled supporting or conflicting, with its date and document id; records the question cannot use are listed as not used.
2. **Flag.** `conflict` if any record is 2 or more scale points from the answer; `disagreement` if a record is 1 point away but on the other side of the satisfactory line (MS vs MU, the line the Bank's outcome statistics use); else `consistent`; `insufficient` if nothing usable.
3. **Confidence.** `high`: the answer comes from the top tier (IEG for outcome; a last ISR younger than 12 months for status) and there is no conflict. `medium`: top tier with a conflict, or a lower-tier direct rating with no disagreement (an ICR not yet validated). `low`: a lower tier contradicted, only ISR proxies for an outcome question, or a stale status.
4. **Status questions compare like with like.** Only ratings of the same (newest) ISR are compared; an older ISR rating that differs is history (`superseded`), not a conflict.

**Why this order.** IEG's rating is independent and made after closing with the full results framework in hand; the ICR is ex post but self-assessed (IEG downgrades a share of them, measured above); the ISR progress rating is a forward-looking judgement made during implementation, and section 6 shows it is the most optimistic. The policy does not throw the optimistic rating away: a conflict is itself information (the agent can say "supervision rated this Satisfactory until closing; IEG later rated it Moderately Unsatisfactory").

## 8. Gold: `proc_gold.project_evidence_resolution`

**What:** `resolve_evidence()` applied to every sampled project for both question types (the evidence of 400 projects is a few thousand rows, so it is collected and resolved in plain Python), stored with the full result as JSON.

**Why:** the agent can read a ready answer with its rationale by SQL, or call the function itself on fresh evidence; both give the same result because both use the same tested code.

In [ ]:
REFERENCE_DATE = datetime.now(timezone.utc).date()
by_project: dict[str, list[dict]] = {}
for r in E.collect():
    by_project.setdefault(r.project_id, []).append(r.asDict())
resolved = []
for pid, recs in by_project.items():
    for qt in HIERARCHY:
        res = resolve_evidence(recs, qt, reference_date=REFERENCE_DATE)
        ans = res["answer"] or {}
        resolved.append({"project_id": pid, "question_type": qt, "answer_rating": ans.get("rating"),
                         "answer_score": ans.get("score"), "answer_source_type": ans.get("source_type"),
                         "answer_as_of": ans.get("as_of"), "answer_doc_id": ans.get("doc_id"), "flag": res["flag"],
                         "confidence": res["confidence"], "n_supporting": len(res["supporting"]),
                         "n_conflicting": len(res["conflicting"]), "rationale": res["rationale"],
                         "resolution_json": json.dumps(res, default=str)})
res_schema = ("project_id string, question_type string, answer_rating string, answer_score int, answer_source_type string, "
              "answer_as_of string, answer_doc_id string, flag string, confidence string, n_supporting int, "
              "n_conflicting int, rationale string, resolution_json string")
res_df = spark.createDataFrame(resolved, res_schema).withColumn("answer_as_of", F.to_date("answer_as_of"))
n_res = publish_table(
    res_df, RESOLUTION_TABLE,
    "Resolved project rating per question type (p09), from proc_silver.project_ratings_evidence via "
    "proc_intel.evidence.resolve_evidence(). Grain: one row per project x question_type.",
    {"project_id": "PK, FK dim_project", "question_type": "PK. outcome or current_status",
     "answer_rating": "Rating chosen by the hierarchy", "answer_score": "Its score (1-6)",
     "answer_source_type": "Source of the answer", "answer_as_of": "Date of the answer's rating",
     "answer_doc_id": "Document the answer cites", "flag": "consistent, disagreement, conflict or insufficient",
     "confidence": "high, medium or low", "n_supporting": "Records supporting the answer",
     "n_conflicting": "Records conflicting with it", "rationale": "Explanation with dates and document ids",
     "resolution_json": "Full resolve_evidence() output"},
    primary_key=["project_id", "question_type"])
add_foreign_key(RESOLUTION_TABLE, ["project_id"], f"{SILVER}.dim_project", ["project_id"])
R = spark.table(RESOLUTION_TABLE)
display(R.groupBy("question_type", "flag", "confidence").count().orderBy("question_type", "flag", "confidence"))

question_type  | flag         | confidence | count
---------------+--------------+------------+------
current_status | conflict     | low        | 1    
current_status | consistent   | high       | 1    
current_status | consistent   | low        | 387  
current_status | disagreement | low        | 11   
outcome        | conflict     | medium     | 58   
outcome        | consistent   | high       | 285  
outcome        | disagreement | high       | 57   

## 9. Demo: ten conflicting projects

**What:** ten projects whose outcome evidence is flagged (conflict or disagreement), disconnect cases first (last ISR satisfactory, IEG unsatisfactory), then hash order, with the ratings side by side and the resolved answer and rationale.

**Why:** shows what the agent would say for exactly the cases where a naive "take the latest document" or "take any rating" would mislead.

In [ ]:
demo = (R.filter("question_type = 'outcome' AND flag IN ('conflict', 'disagreement')")
        .join(wide.select("project_id", "mid_pdo", "isr_pdo", "icr", "ieg"), "project_id")
        .withColumn("_disconnect", (F.col("isr_pdo") >= 4) & (F.col("ieg") < 4))
        .orderBy(F.desc("_disconnect"), F.sha2("project_id", 256)).limit(10)
        .join(spark.table(f"{SILVER}.dim_project").select("project_id", "project_name", "region"), "project_id"))
short_map = F.create_map(*[x for v, abbr in SCORE_SHORT.items() for x in (F.lit(v), F.lit(abbr))])
short = lambda c: F.try_element_at(short_map, F.col(c).cast("int"))
display(demo.select("project_id", F.substring("project_name", 1, 45).alias("project"), "region",
                    short("mid_pdo").alias("mid_ISR"), short("isr_pdo").alias("last_ISR"),
                    short("icr").alias("ICR"), short("ieg").alias("IEG"),
                    "answer_rating", "answer_source_type", "confidence", "rationale").orderBy("project_id"))

project_id | project                                       | region                                               | mid_ISR | last_ISR | ICR  | IEG | answer_rating             | answer_source_type | confidence | rationale                                                                                                                                                                                                                                                                                                                                                                                                                              
-----------+-----------------------------------------------+------------------------------------------------------+---------+----------+------+-----+---------------------------+--------------------+------------+---------------------------------------------------------------------------------------------------------------------------------------------------------

**What:** the full `resolve_evidence()` output for the first demo project, as the agent receives it.

**Why:** the agent's answer template can quote the answer, list the conflicting sources with their dates and links, and carry the confidence label through to the user.

In [ ]:
first = demo.orderBy("project_id").first()
print(json.dumps(json.loads(first.resolution_json), indent=1)[:3500])

{
 "question_type": "outcome",
 "project_id": "P107722",
 "answer": {
  "source_type": "ieg_validation",
  "dimension": "outcome",
  "rating": "Moderately Unsatisfactory",
  "score": 3,
  "as_of": "2019-10-21",
  "independence": "independent",
  "rank": 0,
  "specificity": "direct",
  "doc_id": "31494347",
  "url": "http://documents.worldbank.org/curated/en/540301571671512011",
  "source": "IEG ICR Review, evaluation FY2020"
 },
 "supporting": [
  {
   "source_type": "isr_earlier",
   "dimension": "pdo_progress",
   "rating": "Moderately Unsatisfactory",
   "score": 3,
   "as_of": "2014-09-29",
   "independence": "self_reported",
   "rank": 3,
   "specificity": "proxy",
   "doc_id": "20252357",
   "url": "http://documents.worldbank.org/curated/en/362031468304815837",
   "source": "ISR sequence ?",
   "points_from_answer": 0
  }
 ],
 "conflicting": [
  {
   "source_type": "icr_self_rating",
   "dimension": "outcome",
   "rating": "Satisfactory",
   "score": 5,
   "as_of": "2019-04-29",


## Summary

**What:** the headline numbers as the notebook's exit value.

**Why:** a job run shows them without opening the notebook. (In its own cell: output displayed in the same cell as `dbutils.notebook.exit` is dropped from the run record.)

In [ ]:
dis = rate_row(pdf, "isr_pdo", "last ISR, PDO")
summary = {"projects": len(project_ids), "evidence_rows": n_evidence, "ieg_parse_match": f"{n_match}/{n_check}",
           "last_isr_vs_ieg_projects": dis["projects"], "disconnects": dis["disconnects"],
           "outcome_flags": {r.flag: r["count"] for r in R.filter("question_type = 'outcome'").groupBy("flag").count().collect()}}
print(summary)

{'projects': 400, 'evidence_rows': 2381, 'ieg_parse_match': '381/381', 'last_isr_vs_ieg_projects': 400, 'disconnects': 35, 'outcome_flags': {'disagreement': 57, 'consistent': 285, 'conflict': 58}}


In [ ]:
dbutils.notebook.exit(json.dumps(summary, default=str))

{"projects": 400, "evidence_rows": 2381, "ieg_parse_match": "381/381", "last_isr_vs_ieg_projects": 400, "disconnects": 35, "outcome_flags": {"disagreement": 57, "consistent": 285, "conflict": 58}}